### Imports & Downloads

In [44]:
%pip install uv --quiet
%uv pip install pandas numpy plotly matplotlib
%uv sync

Note: you may need to restart the kernel to use updated packages.


c:\Users\specb\Desktop\School\csc5260\project\Modern-Store-Of-Value\.venv\Scripts\python.exe: No module named pip


Note: you may need to restart the kernel to use updated packages.


Using Python 3.11.15 environment at: C:\Users\specb\Desktop\School\csc5260\project\Modern-Store-Of-Value\.venv
Audited 4 packages in 18ms


Note: you may need to restart the kernel to use updated packages.


Resolved 132 packages in 3ms
Audited 128 packages in 21ms


In [45]:
# Data manipulation tools
import pandas as pd
import datetime

# Visualization tools
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# OS tools
from pathlib import Path
import sys

# Custom Stooq data importer
repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from scripts.stooq_processor import StooqProcessor

### Downloads

In [46]:
stooq_tickers = {
    "Crypto ETFs": [
        "BITW",  # Bitwise 10 Crypto Index
        "IBIT",  # iShares Bitcoin Trust (Replaces BTC-USD)
        "ETHA"   # iShares Ethereum Trust (Replaces ETH-USD)
    ], 
    
    "Individual Stocks": [
        "NVDA", "AAPL", "MSFT", "AMD", "AMZN", "TSLA", "WMT", "LOW", "HD", "JNJ"
    ],
    
    "Sector ETFs": [
        "XLU"    # Utilities Select Sector SPDR Fund
    ],
    
    "Broad Market ETFs": [
        "SPY",   # S&P 500
        "VTI"   # Total US Market (replaces Wilshire 5000)
    ],
    
    "Commodity ETFs (Metals)": [
        "GLD",   # Gold (Baseline)
        "SLV",   # Silver
        "PPLT",  # Platinum
        "PALL"   # Palladium
    ],
    
    "Commodity ETFs (Agriculture)": [
        "WEAT",  # Wheat
        "SOYB",  # Soybeans
        "DBA"    # Broad Agriculture
    ]
}

start_date = "2021-01-01"
end_date = "2026-03-31"

In [47]:
# -----------------------------
# Flatten tickers + category map
# -----------------------------
category_map = {
    ticker: category
    for category, tickers in stooq_tickers.items()
    for ticker in tickers
}

flat_tickers = list(category_map.keys())


# -----------------------------
# Download data
# -----------------------------
with StooqProcessor(repo_root / "data" / "d_us_txt.zip") as processor:

    valid_tickers = [
        t for t in flat_tickers
        if processor.has_ticker(t)
    ]

    missing = sorted(set(flat_tickers) - set(valid_tickers))
    if missing:
        print("Skipping missing tickers:", missing)

    data = processor.download(
        valid_tickers,
        start=start_date,
        end=end_date,
    )


# -----------------------------
# Attach metadata
# -----------------------------
for ticker, frame in data.items():
    data[ticker] = frame.assign(
        Ticker=ticker,
        Category=category_map[ticker],
    )


# -----------------------------
# Combine dataset
# -----------------------------
combined_data = pd.concat(data.values()).reset_index()

data["AAPL"].tail()

,Open,High,Low,Close,Volume,OpenInt,Ticker,Category
Date,,,,,,,,
2025-11-30,270.158,280.380,265.32,278.85,877813393,0,AAPL,Individual Stocks
2025-12-31,278.010,288.620,266.95,271.86,924529010,0,AAPL,Individual Stocks
2026-01-31,272.255,277.840,243.42,259.48,1040017271,0,AAPL,Individual Stocks
2026-02-28,260.030,280.905,255.45,264.18,988633102,0,AAPL,Individual Stocks
2026-03-26,262.410,266.530,246.00,252.89,763091456,0,AAPL,Individual Stocks
